In [ ]:
import json,inspect,time,traceback,math,subprocess,sys,importlib.metadata
from pathlib import Path
url='https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261013-v4/multigpubfs-0.2.0.dev4-py3-none-linux_x86_64.whl'
subprocess.run([sys.executable,'-m','pip','install','multigpubfs[cayleypy] @ '+url],check=True)
import multigpubfs,cayleypy,functools
import numpy as np
np.random.seed(0)
from unittest.mock import patch
import multigpubfs.cayleypy_backend as adapter
from multigpubfs.generic_session import persistent_graph_workers
from multigpubfs.graph_definition import from_cayleypy
multigpubfs.enable_native()
root=Path('/kaggle/working');records=[];failures=[]
print(subprocess.check_output(['nvidia-smi'],text=True),flush=True)
timings=[]
with persistent_graph_workers():
 for n in (3,11,12):
  g=cayleypy.CayleyGraph(cayleypy.PermutationGroups.coxeter(n),device='cpu')
  started=time.perf_counter();r=g.bfs();wall=time.perf_counter()-started
  assert r.bfs_completed and r.num_vertices()==math.factorial(n) and r.diameter()==n*(n-1)//2
  timings.append(dict(n=n,wall_seconds=wall,report=r.report))
  (root/'fixed-coxeter.json').write_text(json.dumps(timings,indent=2));print('COXETER',n,wall,r.report['bfs_seconds'],flush=True)

def oracle_prefix(g,limit=10000):
 seen={tuple(g.start)};frontier={tuple(g.start)};layers=[]
 while frontier:
  layers.append(frontier)
  new={tuple(g.successor(list(x),i)) for x in frontier for i in range(g.generator_count)}-seen
  if len(seen)+len(new)>limit:break
  seen.update(new);frontier=new
 return layers

classes=[('PermutationGroups',cayleypy.PermutationGroups),('MatrixGroups',cayleypy.MatrixGroups),('Puzzles',cayleypy.Puzzles),('MatrixGroupsUnbounded',cayleypy.MatrixGroups)]
special={'koltsov3':{'n':6},'top_spin':{'k':2},'cubic_pancake':{'subset':1},'wrapped_k_cycles':{'k':2},'increasing_k_cycles':{'k':2},'consecutive_k_cycles':{'k':2},'sheveleva2':{'n':5,'k':1},'conjugacy_classes':{'classes':{(2,):None}},'rand_generators':{'k':2},'rubik_cube':{'cube_size':2,'metric':'QTM'},'globe_puzzle':{'a':1,'b':2},'hungarian_rings':{'left_size':4,'left_index':1,'right_size':4,'right_index':1}}
inventory={label:[name for name,fn in inspect.getmembers(cls,inspect.isfunction) if not name.startswith('_')] for label,cls in classes}
(root/'catalogue-inventory.json').write_text(json.dumps(dict(cayleypy=importlib.metadata.version('cayleypy'),factories=inventory),indent=2))
with persistent_graph_workers():
 for label,cls in classes:
  for name in inventory[label]:
   fn=getattr(cls,name);sig=inspect.signature(fn);kw=dict(special.get(name,{}))
   if 'n' in sig.parameters:kw.setdefault('n',3 if label.startswith('MatrixGroups') else 4)
   if 'modulo' in sig.parameters:kw['modulo']=0 if label=='MatrixGroupsUnbounded' else 3
   started=time.perf_counter()
   try:
    definition=fn(**kw);graph=cayleypy.CayleyGraph(definition,device='cpu');g=from_cayleypy(graph)
    oracle=oracle_prefix(g)
    out=root/'catalogue'/label/name
    with patch.object(adapter,'run_graph',functools.partial(multigpubfs.run_graph,capacity=256)):
     result=graph.bfs(output=out,native_options=multigpubfs.NativeOptions(devices=(0,1),max_seconds=20))
    report=result.report
    depth=len(report['layer_sizes'])-1
    assert depth<len(oracle),(name,'CPU_PREFIX_TOO_SHORT',depth,len(oracle))
    assert report['layer_sizes']==[len(x) for x in oracle[:depth+1]],(name,report['layer_sizes'])
    snap=json.loads((out/'states.json').read_text())
    assert len({tuple(x) for x in snap['current']})==len(snap['current'])
    assert all(tuple(x) in oracle[depth] for x in snap['current'])
    record=dict(family=label,factory=name,kwargs=repr(kw),definition=g.to_dict(),status='VERIFIED_COMPLETE' if report['status']=='COMPLETE' else 'VERIFIED_BOUNDED_PREFIX',seconds=time.perf_counter()-started,report=report)
    # Small finite matrices also exercise the public CayleyPy adapter.
    if label=='MatrixGroups':
     started_api=time.perf_counter()
     api=graph.bfs(native_options=multigpubfs.NativeOptions(devices=(0,1),max_seconds=30))
     assert api.bfs_completed and api.layer_sizes==[len(x) for x in oracle]
     record['cayleypy_api']='VERIFIED'
     record['cayleypy_api_seconds']=time.perf_counter()-started_api
     record['cayleypy_api_report']=api.report
   except Exception as e:
    record=dict(family=label,factory=name,kwargs=repr(kw),definition=g.to_dict(),status='FAILED',error=repr(e),traceback=traceback.format_exc());failures.append(record)
   records.append(record);(root/'catalogue-acceptance.json').write_text(json.dumps(records,indent=2));print('FACTORY',label,name,record['status'],flush=True)
assert not failures,[(x['family'],x['factory'],x['error']) for x in failures]
print('VERIFIED_ALL_EXPOSED_FACTORIES',len(records),flush=True)
